### Hybrid Retrival Argumented Generation Evelution using RAGAS 

In [19]:
import warnings 
warnings.filterwarnings('ignore')

# Document load 
from langchain_community.document_loaders import PyPDFLoader 
loader  = PyPDFLoader('Static GK 2025.pdf')
pages = loader.load()

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import hashlib

# Split Data 

spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=180)
text_spliter = spliter.split_documents(pages)
chunks = [i.page_content for i in text_spliter]
metadata = [i.metadata for i in text_spliter]
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]
print(f'print first 5 ids : {ids[:2]}')

print first 5 ids : ['df52eef7bfa55759b4642211e13e3020', '622d6c3b19974d6f39f9950848df1607']


In [21]:
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# client and collection create 
client = chromadb.PersistentClient(path="./Hybrid_RAG")
collection = client.get_or_create_collection(name="Hybrid_RAG",embedding_function=embedding_function)

if chunks:
    collection.add(
        ids=ids,
        documents=chunks , metadatas=metadata
    )
collection.count()

225

In [27]:
# LLM call 
import os 
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
load_dotenv()
try:
    key = os.getenv('GROQ_API_KEY')
    print(bool(key))
except Exception as e:
    print(str(e))
    
Groq = ChatGroq(model="qwen/qwen3.6-27b")

test = Groq.invoke("hello llama?")
test.content

True


'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "hello llama?"\n   - This is a greeting with a question mark, possibly checking if I\'m "Llama" (referring to Meta\'s Llama models).\n\n2.  **Identify Key Elements:**\n   - Greeting: "hello"\n   - Reference: "llama?" (likely referring to Meta\'s LLM family)\n   - Tone: Casual, possibly testing/verifying identity\n\n3.  **Determine My Identity:**\n   - I am Qwen (通义千问), developed by Alibaba Group\'s Tongyi Lab.\n   - I should clearly state my identity while being polite and helpful.\n\n4.  **Formulate Response:**\n   - Acknowledge the greeting\n   - Clarify identity politely\n   - Offer assistance\n   - Keep it concise and friendly\n\n   Draft: "Hello! I\'m actually Qwen, a large language model developed by Alibaba Group\'s Tongyi Lab. How can I help you today?"\n\n5.  **Self-Correction/Refinement:**\n   - Check tone: Friendly and professional\n   - Check accuracy: Correctly identifies as Qwen, not L

In [28]:
# Hybrid Corpus 
from rank_bm25 import BM25Okapi 
def tokenization(token):
    token = token.lower()
    token = token.split()
    return token 

tokens = [tokenization(i) for i in chunks]
bm_corpus = BM25Okapi(tokens)

print(f'sucussfully : {bm_corpus}')

sucussfully : <rank_bm25.BM25Okapi object at 0x13fb129f0>


In [29]:
def Hybrid_Retrive(query:str):
    query_re = Groq.invoke(f"write the query based on symentic search : {query}").content.strip()

    # Thats Vector DB retrival 
    result = collection.query(query_texts=[query_re] , n_results=5)
    dis  = result['distances'][0] 
    docs = result['documents'][0]
    threshold = 0.9
    print(f'the distance is : {dis}')
    dense_docs = []
    for i , d in zip(dis,docs):
        if threshold > i :
            dense_docs.append(d)
    # Thats Hybrid RAG Retrival using indexing 
    query_tokens = tokenization(query_re)
    score = bm_corpus.get_scores(query=query_tokens)
    def get_top_tokens (score , k=10):
        index = list(enumerate(score))
        idx_sorted = sorted(index,key=lambda x:x[1],reverse=True)
        return [doc for doc , _ in idx_sorted[:10]]
    index_tokens = [chunks[i] for i in get_top_tokens(score=score,k=10)]
    
    rrf_token = {}
    
    for rank , doc in enumerate(dense_docs):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_tokens):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
        
    marge = sorted(rrf_token.items() , key = lambda x:x[1] , reverse=True)
    get_docs = [i for i , _ in marge[:5]]
    
    return get_docs
    
def generation_answer(question:str , context_list:list):
    if not context_list :
        return "NOT Related Content"
    content_str = "\n\n".join(context_list) 
    
    prompt = f""" 
    Give answer based on the local document , if cant find out any related content 
    then direct type NOT related content 
    content : {content_str}
    question :{question}
    """
    response = Groq.invoke(prompt)
    
    return response.content

In [ ]:
# RAG evelute 
from datasets import Dataset 
from ragas import evaluate 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from langchain_community.embeddings import HuggingFaceEmbeddings 

user_input = []
retrival_context = []
response = []
reference = []

test_cases = [
    {
        "question": "who is first First Chief of Army Staff",
        "ground_truth": "General Maharaj Rajendra Singh Ji was the first Chief of Army Staff."
    },
    {
        "question": "Where is Malhargad Fort located?",
        "ground_truth": "Malhargad Fort is located in Sonori, near Saswad, Pune district, Maharashtra."
    },
    {
        "question": "Who built the Red Fort in Delhi?",
        "ground_truth": "Red Fort was built by Mughal Emperor Shah Jahan in 1648 AD."
    },
    {
        "question": "What is Purandar Fort famous for?",
        "ground_truth": "Purandar Fort is famous as the birthplace of Chhatrapati Sambhaji Maharaj."
    },
    {
        "question": "Which dynasty built Chitradurga Fort originally?",
        "ground_truth": "Chitradurga Fort was originally built by the Chalukyas between the 11th and 13th centuries."
    },
    {
        "question": "What terms are used in Lawn Tennis?",
        "ground_truth": "Lawn Tennis terms include Ace, Advantage, Backhand Stroke, Baseline, Break Point, Deuce, and Grand Slam."
    },
    {
        "question": "Who is the Libero in Volleyball?",
        "ground_truth": "The Libero is a defensive specialist player in Volleyball."
    },
    {
        "question": "What are the core lifts in Weightlifting?",
        "ground_truth": "The core Olympic weightlifting movements are the Snatch and the Clean and Jerk."
    },
    {
        "question": "Where is Uparkot Fort located?",
        "ground_truth": "Uparkot Fort is located in Junagadh, Gujarat."
    },
    {
        "question": "When was Srirangapatnam Fort built?",
        "ground_truth": "Srirangapatnam Fort was built in 1454 AD by Timmanna Nayaka."
    }
]
for qus , item in enumerate(test_cases):
    q =  item['question']
    truth = item['ground_truth']
    
    context = Hybrid_Retrive(query=q)
    LLM_answer = generation_answer(question=q , context_list=context)
    
    user_input.append(q)
    retrival_context.append(context)
    response.append(LLM_answer)
    reference.append(truth)
    
    data = {
        "user_input":user_input , 
        "retrieved_contexts":retrival_context , 
        "response":response , 
        "reference":reference
    }
    data = Dataset.from_dict(data)

    embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print(LLM_answer)

result = evaluate(
dataset= data, 
metrics=[Faithfulness(),AnswerRelevancy(),ContextPrecision(),ContextRecall()],
embeddings=embedding , 
llm=Groq,
raise_exceptions=False
)

df = result.to_pandas()
print(df)

the distance is : [0.5644669532775879, 0.6074892282485962, 0.6075717806816101, 0.6309012770652771, 0.6409540176391602]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

the distance is : [0.6958768367767334, 0.6968812942504883, 0.6979918479919434, 0.7153205275535583, 0.721405029296875]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

the distance is : [0.6156150102615356, 0.6175653338432312, 0.6243289709091187, 0.6275385022163391, 0.6276595592498779]


...

the distance is : [0.7147214412689209, 0.7229242324829102, 0.7351551651954651, 0.7360914945602417, 0.7391457557678223]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "When was Srirangapatnam Fort built?"
   - **Constraint:** "Give answer based on the local document, if cant find out any related content then direct type NOT related content"
   - **Provided Document:** A collection of snippets from "GK Now – Current Affairs" covering various topics like mountains, forts, dams, books/authors, etc.

2.  **Scan Document for Keywords:**
   - Keywords: "Srirangapatnam", "Fort", "built", "year", "date"
   - I'll search through the provided text for "Srirangapatnam".
   - Scanning...
     - Page 49: Mountains (Rimo I, Teram Kangri, K2, etc.)
     - Page 161: Bhujia Fort, Kangara Fort, Leh Palace
     - Page 40: Dams/Rivers (Indravati, Ranjit Sagar, etc.)
     - Page 152: Neemrana Fort, Lohagarh Fort, Junagarh Fort, Chittorgarh Fort
     - Page 148: Books & Authors (Time Machine, Tom Jones, etc.)
   - Result: "Srirangapatnam" is NOT mentioned anywhere in the provided text.

3

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[0]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[17]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[5]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[13]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in J